# Muse Spark 1.2 Contributor over the FAISS index

Sample RAG call: embed a question, retrieve the closest chunks from a saved FAISS run in `faiss_vectors/`, and ask `muse-spark-1.2-contributor` (Meta Model API) to answer from them only.

- The API key is `MUSE_API` in `backend/.env` (never printed).
- The API is OpenAI-compatible: base URL `https://api.meta.ai/v1`.
- Retrieved chunk text is sent to Meta's API as context.

In [1]:
import os
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "corpus").is_dir())  # repo root, wherever the notebook lives
sys.path.insert(0, str(REPO / "backend"))  # so `src.rag...` imports work

from dotenv import load_dotenv

load_dotenv(REPO / "backend" / ".env")
assert os.getenv("MUSE_API"), "MUSE_API not found in backend/.env"

MODEL = "muse-spark-1.2-contributor"
BASE_URL = "https://api.meta.ai/v1"
RUN_NAME = None  # e.g. "20261029-213539_multilingual-e5-large"; None = newest run

## 1. Load the FAISS run

In [2]:
import numpy as np

from src.rag import embeddings, faiss_store

FAISS_DIR = REPO / "faiss_vectors"
run_dir = FAISS_DIR / RUN_NAME if RUN_NAME else sorted(p for p in FAISS_DIR.iterdir() if p.is_dir())[-1]
index, chunk_records, run_metadata = faiss_store.load_run(run_dir)

# The question must be embedded with the same model that built the index.
assert run_metadata["embedding_model"] == embeddings.model_name(), (
    f"Index was built with {run_metadata['embedding_model']}, "
    f"but EMBEDDING_MODEL is {embeddings.model_name()}"
)
print(run_dir.name, "|", index.ntotal, "vectors |", run_metadata["embedding_dim"], "dims")

20261029-213539_multilingual-e5-large | 359 vectors | 1024 dims


## 2. Retrieve

In [3]:
TOP_K = 5


def retrieve(question, k=TOP_K):
    """Return the ``k`` chunk records closest to the question, each with a ``score``."""
    query_vector = np.array([embeddings.embed_query(question)], dtype="float32")
    scores, positions = index.search(query_vector, k)
    return [
        {**chunk_records[position], "score": float(score)}
        for score, position in zip(scores[0], positions[0])
    ]

## 3. Ask Muse

In [4]:
from openai import BadRequestError, OpenAI

client = OpenAI(base_url=BASE_URL, api_key=os.environ["MUSE_API"])

SYSTEM_PROMPT = (
    "You answer disaster-preparedness questions for people in Metro Manila. "
    "Use ONLY the numbered context passages. Cite passages like [2]. "
    "If the passages do not contain the answer, say you have no verified information "
    "and point to the national emergency hotline 911. Answer in the language of the question."
)


def build_prompt(question, chunks):
    """Put the retrieved chunks in front of the question, numbered so the model can cite them."""
    context = "\n\n".join(f"[{number}] ({chunk['doc_id']}) {chunk['content']}" for number, chunk in enumerate(chunks, 1))
    return f"Context:\n{context}\n\nQuestion: {question}"


def ask(question):
    """Retrieve chunks, call the model, and return (answer, chunks_used).

    The API sometimes rejects a prompt with content_policy_violation even though
    the text is harmless disaster-preparedness material. When that happens the call
    is retried with fewer chunks (the best-ranked ones are kept).
    """
    chunks = retrieve(question)
    for count in range(len(chunks), 0, -1):
        used = chunks[:count]
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_prompt(question, used)},
                ],
            )
            return response.choices[0].message.content, used
        except BadRequestError as error:
            if "content_policy_violation" not in str(error):
                raise
            print(f"  (filtered with {count} chunks, retrying with {count - 1})")
    return "The API refused every version of this prompt.", []

## 4. Sample questions

In [5]:
SAMPLE_QUESTIONS = [
    "Who do I call for a flood rescue in Marikina?",
    "What should go in an emergency go-bag?",
    "Sino ang puwedeng mag-lift ng state of imminent disaster?",
    "What is the elevation of the sampled locations in Manila?",
]

for question in SAMPLE_QUESTIONS:
    answer, chunks = ask(question)
    print("Q:", question)
    print("A:", answer)
    print(f"Chunks used: {len(chunks)}")
    print("Sources:", ", ".join(f"[{n}] {c['id']} ({c['score']:.2f})" for n, c in enumerate(chunks, 1)))
    print("-" * 80)

/home/kenneth28/my_projects/Flood-Gentic/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:  16%|█▌        | 62/391 [00:00<00:00, 618.76it/s]

Loading weights:  32%|███▏      | 124/391 [00:00<00:00, 555.12it/s]

Loading weights:  47%|████▋     | 182/391 [00:00<00:00, 563.47it/s]

...

  (filtered with 5 chunks, retrying with 4)


  (filtered with 4 chunks, retrying with 3)


  (filtered with 3 chunks, retrying with 2)


Q: Sino ang puwedeng mag-lift ng state of imminent disaster?
A: Ayon sa IRR, ang may awtoridad na mag-lift ng State of Imminent Disaster ay:

a. Ang **Pangulo (President)** sa national level, sa rekomendasyon ng NDRRMC [2]
b. Ang **Local Chief Executive (LCE)** sa local level, sa rekomendasyon ng RDRRMC [2]

Maaaring i-lift ang deklarasyon kapag nangyari na ang forecasted hazard o kapag hindi natugunan ang projected impacts sa loob ng allowable lead time batay sa pre-disaster risk assessment [1].
Chunks used: 2
Sources: [1] 20260914-NDRRMC-IRR-RA12287:4 (0.84), [2] 20260914-NDRRMC-IRR-RA12287:5 (0.84)
--------------------------------------------------------------------------------


Q: What is the elevation of the sampled locations in Manila?
A: Based on the provided records, the elevation of the sampled locations in Manila is:

* Manila location 2: 9 m [2]
* Manila location 5: 9 m [4]
* Manila location 4: 15 m [1]
* Manila location 10: 15 m [5]
* Manila location 14: 29 m [3]
Chunks used: 5
Sources: [1] ncr-locations-manila:1 (0.85), [2] ncr-locations-manila:0 (0.85), [3] ncr-locations-manila:4 (0.85), [4] ncr-locations-manila:2 (0.85), [5] ncr-locations-manila:3 (0.85)
--------------------------------------------------------------------------------
